# Solution: Trivy-Based Supply Chain Assessment for a RAG Inference Platform

This notebook completes the assessment workflow by parsing Trivy findings, tracing SBOM dependency chains, prioritizing operational risk, and exporting the final report.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from rag_supply_chain_utils import (
    build_assessment,
    extract_misconfigurations,
    extract_vulnerabilities,
    format_console_table,
    load_json,
    render_report,
    summarize_sbom_components,
    validate_trivy_report,
    write_csv,
    write_json,
)

report_path = ROOT / "data" / "trivy_rag_platform_report.json"
table_path = ROOT / "data" / "trivy_rag_platform_table.txt"
sbom_path = ROOT / "data" / "rag_platform_sbom.cdx.json"
results_dir = ROOT / "results"

## 1. Inspect the Trivy reports

In [ ]:
print(table_path.read_text(encoding="utf-8")[:1600])
report = load_json(report_path)
validate_trivy_report(report)

## 2. Extract vulnerabilities and container misconfigurations

In [ ]:
vulnerabilities = extract_vulnerabilities(report)
misconfigurations = extract_misconfigurations(report)
[(item.vulnerability_id, item.package, item.severity) for item in vulnerabilities[:5]], len(misconfigurations)

## 3. Review SBOM components and dependency relationships

In [ ]:
sbom = load_json(sbom_path)
summarize_sbom_components(sbom)[:8]

## 4. Prioritize findings by operational exposure

In [ ]:
assessment = build_assessment(report, sbom, vulnerabilities, misconfigurations)
print(format_console_table(assessment["prioritized_findings"]))

## 5. Interpret SBOM dependency chains

In [ ]:
assessment["dependency_chain_analysis"][:5]

## 6. Export deliverables

In [ ]:
results_dir.mkdir(parents=True, exist_ok=True)
write_json(assessment, results_dir / "rag_supply_chain_assessment.json")
write_csv(assessment["prioritized_findings"], results_dir / "categorized_vulnerabilities.csv")
write_csv(assessment["dependency_chain_analysis"], results_dir / "sbom_dependency_chain_analysis.csv")
write_csv(assessment["sbom_components"], results_dir / "sbom_component_summary.csv")
(results_dir / "ai_supply_chain_assessment_report.md").write_text(render_report(assessment), encoding="utf-8")
sorted(path.name for path in results_dir.iterdir())